# BEATs Core-2 checkpoint-selection sensitivity

本 Notebook 只读取 repaired R0/N1/A1/L1/L2 的 `train_log.jsonl`，不读取 test、不运行模型，也不改变现有 selected checkpoint。静态报告见 `reproduce/pafa/CHECKPOINT_SELECTION_SENSITIVITY.md`。


## 比较口径

1. `current equal raw loss`：每 dataset 等权 mean eligible Level1/Crackle/Wheeze plain CE/BCE，再对 ICBHI/SPR 等权。
2. `e1-normalized mean`：每 dataset loss 除以本数据集 epoch-1 loss后等权平均。
3. `normalized worst`：最小化两个 epoch-1-relative losses 中较差者。
4. `prospective native composite`：未来预注册，在 validation 内用互斥 group-safe calibration/selection；calibration 拟合 shared attributes thresholds，selection 以 ICBHI Score 与 SPR Task1-1 Score 等权均值选 epoch。

后两种 loss normalization 只是 sensitivity；native composite 只是未来合同。任何 alternative epoch checkpoint 缴失都保持 HOLD，现有五条不得事后重选。


In [ ]:
from pathlib import Path
from baseline.multidataset_pipeline.checkpoint_selection_sensitivity import analyze_root

EXECUTE_READ_ONLY = False
LOGS_ROOT = Path('result/reproduce/beats_nal_ablation')
analysis = analyze_root(LOGS_ROOT) if EXECUTE_READ_ONLY else None
analysis


## 解释边界与 prospective split

现有完整 train logs 足以重算 loss-only criteria，但本地没有所有 epoch 的 predictions/checkpoints。不能从一个 log row 恢复模型，也不能用现有 terminal/test 结果决定 criterion。

前瞻方案应先冻结 validation group subdivision：calibration 与 selection group 互斥；threshold 只由 calibration 获得；checkpoint 只由 selection native composite 获得；tie 取 earliest epoch。epoch 固定后才可在 full validation 按同一规则冻结 threshold，再访问 test 一次。

**Test Result: Not run. Decision: design sensitivity complete；任何重训、服务器读取或 alternative checkpoint terminal evaluation 均需用户另批。**
